# Task 2 — Model Building & Training

We train and evaluate two models on **both** datasets:

- **Logistic Regression** — interpretable baseline.
- **XGBoost** — gradient-boosted ensemble.

Key choices for the imbalanced setting:
- Stratified train/test split.
- Preprocessing (+ optional SMOTE) wrapped in an imbalanced-learn `Pipeline`
  so resampling only touches the training fold (no leakage).
- Primary metrics: **AUC-PR** and **F1**, plus the confusion matrix.

> Run Task 1 first (or `python -m src.pipeline`) so the cleaned data exists.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src import config, transform, modeling, evaluation, pipeline

pd.set_option("display.max_columns", 50)

## 1. Load processed data

Load `data/processed/*.csv` if present, otherwise rebuild from raw.

In [ ]:
fraud_path = config.PROCESSED_DIR / "fraud_processed.csv"
cc_path = config.PROCESSED_DIR / "creditcard_processed.csv"

if fraud_path.exists():
    fraud = pd.read_csv(fraud_path)
else:
    fraud = pipeline.build_fraud_dataset()

if cc_path.exists():
    cc = pd.read_csv(cc_path)
else:
    cc = pipeline.build_creditcard_dataset()

print("Fraud_Data:", fraud.shape, "| creditcard:", cc.shape)

## 2. E-commerce dataset (`Fraud_Data`)

### 2.1 Split & preprocessor

In [ ]:
target = "class"
X_train, X_test, y_train, y_test = transform.stratified_split(fraud, target)
num_cols, cat_cols = transform.split_feature_types(fraud, target)
pre = transform.build_preprocessor(num_cols, cat_cols)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train fraud rate:", round(y_train.mean(), 4))

### 2.2 Train Logistic Regression & XGBoost

In [ ]:
logreg = modeling.build_logreg_pipeline(pre, use_smote=True)
logreg.fit(X_train, y_train)

xgb = modeling.build_xgb_pipeline(pre, y_train, use_smote=False)
xgb.fit(X_train, y_train)

results = [
    evaluation.evaluate(logreg, X_test, y_test, "LogReg (e-commerce)"),
    evaluation.evaluate(xgb, X_test, y_test, "XGBoost (e-commerce)"),
]

In [ ]:
evaluation.plot_confusion(logreg, X_test, y_test, "logreg_ecommerce")
evaluation.plot_confusion(xgb, X_test, y_test, "xgb_ecommerce")
evaluation.plot_pr_curve(
    {"LogReg": logreg, "XGBoost": xgb}, X_test, y_test, name="pr_ecommerce"
)
plt.show()
evaluation.metrics_table(results)

## 3. Bank dataset (`creditcard`)

Same workflow; this dataset is extremely imbalanced (~0.17% fraud).

In [ ]:
target_cc = "Class"
Xtr_c, Xte_c, ytr_c, yte_c = transform.stratified_split(cc, target_cc)
num_c, cat_c = transform.split_feature_types(cc, target_cc)
pre_c = transform.build_preprocessor(num_c, cat_c)

logreg_c = modeling.build_logreg_pipeline(pre_c, use_smote=True).fit(Xtr_c, ytr_c)
xgb_c = modeling.build_xgb_pipeline(pre_c, ytr_c, use_smote=False).fit(Xtr_c, ytr_c)

results_cc = [
    evaluation.evaluate(logreg_c, Xte_c, yte_c, "LogReg (creditcard)"),
    evaluation.evaluate(xgb_c, Xte_c, yte_c, "XGBoost (creditcard)"),
]

In [ ]:
evaluation.plot_confusion(logreg_c, Xte_c, yte_c, "logreg_creditcard")
evaluation.plot_confusion(xgb_c, Xte_c, yte_c, "xgb_creditcard")
evaluation.plot_pr_curve(
    {"LogReg": logreg_c, "XGBoost": xgb_c}, Xte_c, yte_c, name="pr_creditcard"
)
plt.show()
evaluation.metrics_table(results_cc)

## 4. Save the best models

In [ ]:
import joblib

models_dir = config.PROJECT_ROOT / "models"
models_dir.mkdir(exist_ok=True)
joblib.dump(xgb, models_dir / "xgb_ecommerce.joblib")
joblib.dump(xgb_c, models_dir / "xgb_creditcard.joblib")

# Combined comparison across both datasets.
evaluation.metrics_table(results + results_cc)